# Chapter 23 — Time Series II: Monetary Transmission

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch23_monetary_transmission_var.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 23.1 Building a monthly dataset

In [ ]:
import re
import html
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM

from indiadata import bis, mospi_cpi, mospi_iip, oecd_rates

iip_raw = mospi_iip(base_year="2011-12")
iip = iip_raw[iip_raw["type"] == "General"].set_index("date")[["index", "growth_rate"]]
cpi = mospi_cpi().set_index("date")[["index", "inflation"]]
repo = bis("WS_CBPOL", "M.IN").set_index("date")["value"]
neer = bis("WS_EER", "M.N.B.IN").set_index("date")["value"]
rates = oecd_rates().set_index("date")

for name, s in [("IIP", iip), ("CPI", cpi), ("repo", repo), ("NEER", neer),
                ("OECD rates", rates)]:
    print(f"{name:10s} {s.index.min():%b %Y} – {s.index.max():%b %Y}  ({len(s)} months)")

In [ ]:
def yoy(level):
    """Year-on-year change in per cent, as 100 × 12-month log difference."""
    return 100 * np.log(level).diff(12)

data = pd.concat({"ip": yoy(iip["index"]), "infl": yoy(cpi["index"]),
                  "repo": repo, "neer": yoy(neer),
                  "short": rates["short_rate"], "long": rates["long_rate"]},
                 axis=1, sort=True).loc["2014-01":"2025-12"]
data.index.freq = "MS"
print(data.shape, "| missing values:", int(data.isna().sum().sum()))
data.describe().loc[["mean", "std", "min", "max"]].round(2)

In [ ]:
pct = pd.DataFrame({"IIP: ours (%)": iip["index"].pct_change(12) * 100,
                    "IIP: MoSPI": iip["growth_rate"],
                    "CPI: ours (%)": cpi["index"].pct_change(12) * 100,
                    "CPI: MoSPI": cpi["inflation"]}).loc["2014-01":"2025-12"]
print("Largest gap, IIP:", round((pct.iloc[:, 0] - pct.iloc[:, 1]).abs().max(), 2),
      "| CPI:", round((pct.iloc[:, 2] - pct.iloc[:, 3]).abs().max(), 2))
print("Months with no published figure:")
print(pct[pct.isna().any(axis=1)].round(1))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
panels = [("ip", "IIP growth, y-o-y %", "#1f4e79"),
          ("infl", "CPI inflation, y-o-y %", "#c55a11"),
          ("repo", "Repo rate, %", "#1f4e79"),
          ("neer", "NEER, y-o-y % (+ = appreciation)", "#c55a11")]
for ax, (col, title, colour) in zip(axes.flat, panels):
    ax.plot(data.index, data[col], color=colour, lw=1.4)
    ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"),
               color="grey", alpha=0.15)
    ax.set_title(title, loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].set_ylim(-40, 40)
axes[0, 0].text(pd.Timestamp("2020-06-01"), -37, "cut off: −85 and +85",
                fontsize=8, color="grey")
fig.suptitle("The four variables of the VAR, 2014–2025", x=0.01, ha="left",
             fontsize=13)
fig.text(0.01, -0.02, "Grey: March 2020–June 2021 (COVID-19). Sources: MoSPI (IIP base "
         "2011-12, CPI base 2012), BIS policy rates and effective exchange rates.",
         fontsize=8, color="grey")
plt.tight_layout()
plt.show()

In [ ]:
print("IIP growth, lowest:", data["ip"].idxmin().strftime("%b %Y"),
      round(data["ip"].min(), 1), "| highest:", data["ip"].idxmax().strftime("%b %Y"),
      round(data["ip"].max(), 1))
print("Repo rate range:", data["repo"].min(), "to", data["repo"].max())
print("Repo rate in December of each year:",
      data["repo"][data.index.month == 12].set_axis(range(2014, 2026)).to_dict())

## 23.2 What to do about COVID-19

In [ ]:
covid_months = pd.to_datetime(["2020-03-01", "2020-04-01", "2020-05-01", "2020-06-01",
                               "2021-03-01", "2021-04-01", "2021-05-01", "2021-06-01"])
dummies = pd.DataFrame({f"d_{m:%Y_%m}": (data.index == m).astype(float)
                        for m in covid_months}, index=data.index)
print(dummies.sum().to_dict())

## 23.3 A small VAR: lag length and estimation

In [ ]:
VARS = ["ip", "infl", "repo", "neer"]          # the Cholesky ordering (23.5)
model = VAR(data[VARS], exog=dummies)
sel = model.select_order(maxlags=12)
print(sel.selected_orders)

In [ ]:
P = sel.selected_orders["aic"]
res = model.fit(P)
print(f"VAR({P}): {res.nobs} months used, {res.params.shape[0]} coefficients per equation")
print(f"Stable: {res.is_stable()};  largest eigenvalue modulus "
      f"{1 / np.abs(res.roots).min():.3f}")

In [ ]:
white = res.test_whiteness(nlags=12, adjusted=True)
print(f"Portmanteau test, 12 lags: statistic {white.test_statistic:.1f}, "
      f"p-value {white.pvalue:.3f}")

## 23.4 Granger causality

In [ ]:
tests = [("repo", "infl"), ("repo", "ip"), ("infl", "repo"), ("ip", "repo"),
         ("neer", "infl")]
gc = pd.DataFrame([{"does": x, "Granger-cause": y,
                    "F": res.test_causality(y, [x], kind="f").test_statistic,
                    "p-value": res.test_causality(y, [x], kind="f").pvalue}
                   for x, y in tests])
gc.round(3)

## 23.5 Impulse responses and the Cholesky ordering

In [ ]:
H = 36
irf = res.irf(H)
k_repo, k_infl, k_ip = VARS.index("repo"), VARS.index("infl"), VARS.index("ip")
sd_repo = np.sqrt(res.sigma_u.loc["repo", "repo"])
scale = 0.25 / sd_repo
print(f"One-SD repo shock = {100 * sd_repo:.1f} basis points; responses scaled by {scale:.2f}")


def response(irf_obj, resp, imp, scale):
    """Orthogonalised response with ±1.96 asymptotic standard errors, rescaled."""
    point = irf_obj.orth_irfs[:, resp, imp] * scale
    se = irf_obj.stderr(orth=True)[:, resp, imp] * scale
    return pd.DataFrame({"response": point, "low": point - 1.96 * se,
                         "high": point + 1.96 * se})

r_infl = response(irf, k_infl, k_repo, scale)
r_ip = response(irf, k_ip, k_repo, scale)
r_repo = response(irf, k_repo, k_repo, scale)
r_infl.iloc[[0, 3, 6, 9, 12, 18, 24, 36]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.8))
for ax, (r, title) in zip(axes, [(r_repo, "Repo rate"), (r_infl, "CPI inflation"),
                                 (r_ip, "IIP growth")]):
    ax.fill_between(r.index, r["low"], r["high"], color="#9db7d5", alpha=0.5, lw=0,
                    label="95% band")
    ax.plot(r.index, r["response"], color="#1f4e79", lw=2, label="Response")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(title, loc="left", fontsize=11)
    ax.set_xlabel("Months after the hike")
    ax.set_xticks([0, 12, 24, 36])
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Percentage points")
axes[0].legend(frameon=False, fontsize=8)
fig.suptitle("Responses to a 25 bp surprise repo hike (VAR with COVID dummies)",
             x=0.01, ha="left", fontsize=13)
fig.text(0.01, -0.04, f"VAR({P}) in IIP growth, CPI inflation, repo, NEER change; "
         "Cholesky ordering in that order; 2014–2025 with COVID dummies. "
         "Asymptotic 95% bands.", fontsize=8, color="grey")
plt.tight_layout()
plt.show()

In [ ]:
peak = r_infl["response"].idxmin()
sig = r_infl[r_infl["high"] < 0]
print(f"Peak fall in inflation: {r_infl.loc[peak, 'response']:.3f} points, "
      f"{peak} months after the hike")
print("Months in which the 95% band lies wholly below zero:",
      list(sig.index) if len(sig) else "none")
print(f"Repo rate: highest {r_repo['response'].max():.2f} points (month "
      f"{r_repo['response'].idxmax()}); still {r_repo.loc[12, 'response']:.2f} after 12 months")
print(f"IIP growth: lowest {r_ip['response'].min():.2f} points (month "
      f"{r_ip['response'].idxmin()}); months with band below zero: "
      f"{list(r_ip.index[r_ip['high'] < 0]) or 'none'}")

## 23.6 Why the ordering matters

In [ ]:
ALT = ["repo", "ip", "infl", "neer"]
res_alt = VAR(data[ALT], exog=dummies).fit(P)
irf_alt = res_alt.irf(H)
scale_alt = 0.25 / np.sqrt(res_alt.sigma_u.loc["repo", "repo"])
r_alt = response(irf_alt, ALT.index("infl"), ALT.index("repo"), scale_alt)
comp = pd.DataFrame({"repo ordered 3rd": r_infl["response"],
                     "repo ordered 1st": r_alt["response"]})
comp.iloc[[0, 1, 3, 6, 9, 12, 24]].round(3)

In [ ]:
corr = res.resid.corr().loc["repo", ["ip", "infl"]]
print("Correlation of the reduced-form repo shock with the shocks to:",
      corr.round(2).to_dict())

## 23.7 Forecast-error variance decomposition

In [ ]:
fevd = res.fevd(24)
fe_infl = pd.DataFrame(fevd.decomp[k_infl], columns=[f"{v} shock" for v in VARS],
                       index=range(1, 25))
(100 * fe_infl.loc[[1, 3, 6, 12, 24]]).round(1)

## 23.8 Sensitivity to the COVID treatment

In [ ]:
def infl_response(df, exog=None):
    r = VAR(df[VARS], exog=exog).fit(P)
    sc = 0.25 / np.sqrt(r.sigma_u.loc["repo", "repo"])
    return response(r.irf(H), k_infl, k_repo, sc), r.nobs

variants = {"With COVID dummies (2014–2025)": infl_response(data, dummies),
            "Pre-COVID only (2014–Feb 2020)": infl_response(data.loc[:"2020-02"]),
            "No treatment (2014–2025)": infl_response(data)}
summary = pd.DataFrame({name: {"months used": n,
                               "peak response": r["response"].min(),
                               "peak month": r["response"].idxmin(),
                               "response at 12 m": r.loc[12, "response"],
                               "response at 24 m": r.loc[24, "response"],
                               "band excludes 0 (months)": int((r["high"] < 0).sum())}
                        for name, (r, n) in variants.items()}).T
summary.astype(float).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
styles = {"With COVID dummies (2014–2025)": ("#1f4e79", "-"),
          "Pre-COVID only (2014–Feb 2020)": ("#c55a11", "-"),
          "No treatment (2014–2025)": ("#9db7d5", "--")}
for name, (r, n) in variants.items():
    colour, ls = styles[name]
    ax.plot(r.index, r["response"], color=colour, ls=ls, lw=2, label=name)
r0 = variants["With COVID dummies (2014–2025)"][0]
ax.fill_between(r0.index, r0["low"], r0["high"], color="#1f4e79", alpha=0.1, lw=0)
ax.axhline(0, color="black", lw=0.8)
ax.axvspan(9, 12, color="green", alpha=0.08)
ax.text(9.2, ax.get_ylim()[1] * 0.85, "RBI view:\n3–4 quarters", fontsize=8,
        color="green")
ax.set_xticks([0, 6, 12, 18, 24, 30, 36])
ax.set_xlabel("Months after a 25 bp surprise hike")
ax.set_ylabel("Response of CPI inflation, pp")
ax.set_title("Inflation response under three treatments of COVID-19", loc="left",
             fontsize=13)
ax.legend(frameon=False, fontsize=9, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.05, "Shaded blue: 95% band for the dummy version. Green: lag of 3–4 "
         "quarters cited by the RBI (section 23.9).", fontsize=8, color="grey")
plt.show()

## 23.9 What does the RBI say?

In [ ]:
RBI_URL = "https://www.rbi.org.in/Scripts/BS_ViewBulletin.aspx?Id=17255"
cache = Path("data/raw/rbi_bulletin_17255.txt")
if not cache.exists():
    page = requests.get(RBI_URL, timeout=60,
                        headers={"User-Agent": "python-for-indian-economics (teaching use)"})
    page.raise_for_status()
    text = html.unescape(re.sub(r"<[^>]+>", " ", page.text))
    cache.write_text(re.sub(r"\s+", " ", text), encoding="utf-8")
text = cache.read_text(encoding="utf-8")

title = re.search(r"Monetary Transmission in India[^-]*- Viral V Acharya Date : \w+ \d+, \d{4}", text)
lag = re.search(r"[^.]*lag of 2-3 quarters on output[^.]*\.", text)
print(title.group(0) if title else "title not found")
print(lag.group(0).strip() if lag else "sentence not found")

## 23.10 Cointegration: the repo rate and the 10-year yield

In [ ]:
lv = data[["repo", "long"]]
spread = lv["long"] - lv["repo"]
warnings.simplefilter("ignore", FutureWarning)   # adfuller's notice about its return type
for name, x in [("repo", lv["repo"]), ("10-year yield", lv["long"]),
                ("spread (10y − repo)", spread)]:
    print(f"ADF p-value, {name:20s}: {adfuller(x, autolag='AIC')[1]:.3f}")
print(f"Spread: mean {spread.mean():.2f}, min {spread.min():.2f} "
      f"({spread.idxmin():%b %Y}), max {spread.max():.2f} ({spread.idxmax():%b %Y})")

In [ ]:
warnings.simplefilter("ignore", np.exceptions.ComplexWarning)  # harmless rounding inside statsmodels
rows = []
for det, label in [(-1, "no constant"), (0, "constant"), (1, "constant + trend")]:
    j = coint_johansen(lv, det_order=det, k_ar_diff=2)
    for r0 in (0, 1):
        rows.append({"deterministic": label, "H0": f"r <= {r0}",
                     "trace stat": j.lr1[r0], "5% critical": j.cvt[r0, 1],
                     "reject at 5%": j.lr1[r0] > j.cvt[r0, 1]})
pd.DataFrame(rows).round(2)

In [ ]:
vecm = VECM(lv, k_ar_diff=2, coint_rank=1, deterministic="ci").fit()
beta = vecm.beta.ravel()
const = vecm.const_coint[0, 0]       # ec = repo + beta[1] * yield + const
print(f"Long-run relation: repo = {-beta[1]:.2f} × yield − {const:.2f}")
alpha = pd.DataFrame({"alpha": vecm.alpha.ravel(), "z": vecm.alpha.ravel() / vecm.stderr_alpha.ravel()},
                     index=["repo equation", "yield equation"])
alpha.round(3)

## 23.11 What a small VAR can and cannot tell us

## Key ideas

## Exercises

## Data citation

## Further reading